In [7]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [8]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
red_shapefiles = os.path.join(folder,"red_shapefiles")

#### 1. Read Connectors GPKG (from algorithm)

In [10]:
connectors = gpd.read_file(os.path.join(red_shapefiles,"10_red_final_corrected","connectors_corrected.gpkg"))
zones_standarized = gpd.read_file(os.path.join(red_shapefiles,"zonas_agebs_standarized.shp"))

connectors = connectors.merge(
    zones_standarized[['clave_ageb', 'id_mun_age']],
    on='clave_ageb',
    how='left'
)

connectors = connectors.rename(columns={'id_mun_age': 'zone_id'})
connectors = connectors.sort_values(by='zone_id').reset_index(drop=True)

In [11]:
connectors

,clave_ageb,link_id,node_id,node_position,main_road,highway,candidate_level,node_highways,centroid_distance_m,angle_deg,number_of_connectors,selection_order,selected_connectors,connector_length_m,geometry,zone_id
0,1403900010026,439506,185568,to,2,secondary,1,"secondary,service",466.752791,38.819997,5,1,5,466.752791,"LINESTRING (2356459.998 967475.311, 2356823.65...",1
1,1403900010026,470522,160984,from,2,secondary,1,"residential,secondary",596.484728,0.121340,5,2,5,596.484728,"LINESTRING (2356459.998 967475.311, 2357056.48...",1
2,1403900010026,376452,133628,from,4,primary_link,1,"primary,primary_link,residential",494.181773,161.660590,5,3,5,494.181773,"LINESTRING (2356459.998 967475.311, 2355990.91...",1
3,1403900010026,536356,185360,to,12,residential,3,"residential,service",249.501008,262.109739,5,4,5,249.501008,"LINESTRING (2356459.998 967475.311, 2356425.74...",1
4,1403900010026,383194,161064,to,12,residential,3,"['living_street', 'footway'],residential",250.782168,309.971734,5,5,5,250.782168,"LINESTRING (2356459.998 967475.311, 2356621.10...",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9614,1412401760599,68508,21363,to,12,residential,3,residential,86.537808,268.402153,4,4,4,86.537808,"LINESTRING (2387026.885 946235.02, 2387024.472...",9052
9615,1412401760599,68516,26147,to,11,unclassified,3,"residential,unclassified",162.495130,141.798012,4,1,4,162.495130,"LINESTRING (2387026.885 946235.02, 2386899.19 ...",9052
9616,1412401760599,68508,21360,from,12,residential,3,residential,55.408655,324.460530,4,2,4,55.408655,"LINESTRING (2387026.885 946235.02, 2387071.971...",9052
9617,1412401760601,34018,29441,to,12,residential,3,"residential,service",74.797435,47.152448,2,2,2,74.797435,"LINESTRING (2387970.089 947035.446, 2388020.95...",9053


#### Create connector objects in Visum

In [12]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = os.path.join(folder, "Red Base GDL", "RedBase 300726 - conectores_final.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [13]:
for row in connectors.itertuples():
    zone_no = row.zone_id
    node_no = row.node_id

    try:
        # Create connector from zone to node
        
        Visum.Net.AddConnector(zone_no, node_no)

        # Get both direction connectors
        #source_connector = Visum.Net.Connectors.SourceItemByKey(zone_no, node_no)
        #destination_connector = Visum.Net.Connectors.DestItemByKey(node_no, zone_no)

        # Set TSysSet to "C" for both connectors
        #source_connector.SetAttValue("TSysSet", "C")
        #destination_connector.SetAttValue("TSysSet", "C")
    except Exception as e:
        print(f"Error creating connector for zone {zone_no} and node {node_no}: {e}")